## Cell 01

# Station Project 1: The Prediction
## Online Payments Fraud Detection

**Goal:** learn to classify a transaction as fraud (`1`) or not fraud (`0`) using
the synthetic PaySim dataset. This is supervised binary classification: training
examples contain both transaction details and the correct label.

Run every cell in order with **Kernel → Restart Kernel and Run All Cells**.
The first run downloads a large CSV and may take several minutes and substantial
memory. All tables, charts, metrics and the illustrative prediction stay visible.

**Assignment coverage:** this notebook implements the twelve requirements supplied
in the request. The original assignment attachment was not accessible in the
workspace, so comparison with any additional wording in that document is pending.
The two peer reviews and GitHub submission remain manual steps (see README).

## Cell 02

### Environment setup
`%pip` installs into this notebook's Python environment and fixes the original
`ModuleNotFoundError`. If Jupyter asks for a kernel restart after installation,
restart it and run all cells again. `nbclient` supports the optional automated run.

In [1]:
# Cell 03
%pip install kagglehub pandas numpy plotly scikit-learn jupyterlab nbformat nbclient ipykernel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 04

### Import libraries and choose reproducible settings
Pandas manages tables, NumPy handles arrays, Plotly makes interactive charts,
and scikit-learn provides splitting, preprocessing, training and evaluation.
`Path` locates files portably; `json` saves readable results. Package versions
are saved with the results because different versions can change behavior.

In [2]:
# Cell 05
from pathlib import Path
import json
from importlib.metadata import version

import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

RANDOM_SEED = 42
DATASET_HANDLE = "ealaxi/paysim1"
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
package_versions = {
    name: version(name)
    for name in ["kagglehub", "pandas", "numpy", "plotly", "scikit-learn"]
}
display(pd.Series(package_versions, name="Installed version").to_frame())


C:\Users\jonat\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Installed version
kagglehub,1.0.3
pandas,3.0.2
numpy,2.4.3
plotly,6.7.0
scikit-learn,1.9.1


## Cell 06

### Download and load the dataset
Source: [PaySim on Kaggle](https://www.kaggle.com/datasets/ealaxi/paysim1).
PaySim simulates mobile money transactions; it is not real customers' payment data.
KaggleHub returns its download/cache folder, which we search recursively for a CSV
with the required columns. No machine-specific path is used. Internet access and,
if Kaggle requests it, Kaggle authentication are needed for the download.
We keep the full dataset; we do not silently substitute the workspace CSV.

In [3]:
# Cell 07
dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
print("Dataset folder:", dataset_path)
required_columns = {
    "step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"
}
csv_candidates = sorted(dataset_path.rglob("*.csv"))
matching_csvs = [
    path for path in csv_candidates
    if required_columns.issubset(pd.read_csv(path, nrows=0).columns)
]
if len(matching_csvs) != 1:
    raise ValueError(f"Expected one PaySim CSV; found {len(matching_csvs)}: {matching_csvs}")
csv_path = matching_csvs[0]
# Small categorical/label types save memory without rounding monetary values.
df = pd.read_csv(csv_path, dtype={
    "type": "category", "isFraud": "int8", "isFlaggedFraud": "int8"
})
print("Loaded CSV:", csv_path.name)
print("Rows and columns:", df.shape)
display(df.head())



  0%|          | 0.00/178M [00:00<?, ?B/s]


  1%|          | 1.00M/178M [00:02<06:20, 487kB/s]


  1%|          | 2.00M/178M [00:02<03:38, 843kB/s]


  2%|▏         | 3.00M/178M [00:03<02:32, 1.20MB/s]


  2%|▏         | 4.00M/178M [00:03<02:03, 1.48MB/s]


  3%|▎         | 5.00M/178M [00:04<01:54, 1.59MB/s]

... (163 outputs left out)

Extracting files...


Dataset folder: C:\Users\jonat\.cache\kagglehub\datasets\ealaxi\paysim1\versions\2


Loaded CSV: PS_20174392719_1491204439457_log.csv
Rows and columns: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Cell 08

### Inspect the data before modeling
Check sample rows, dimensions, data types, missing values, exact duplicate rows,
and class counts. A missing target cannot teach a model the correct answer, so
we reject such data instead of inventing labels. Missing predictors will be
imputed later using training data only. Zero balances are retained: zero is a
valid value, and some destination balances may be unavailable in the simulation.

In [4]:
# Cell 09
print("Shape:", df.shape)
display(df.sample(n=min(5, len(df)), random_state=RANDOM_SEED))
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum()
}))
duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows:", duplicate_count)
if df["isFraud"].isna().any() or not df["isFraud"].isin([0, 1]).all():
    raise ValueError("isFraud must contain only known binary labels 0 and 1.")
fraud_counts = df["isFraud"].value_counts().reindex([0, 1], fill_value=0)
fraud_table = pd.DataFrame({
    "label": ["Not fraud", "Fraud"],
    "count": fraud_counts.to_numpy(),
    "percentage": fraud_counts.to_numpy() / len(df) * 100
})
display(fraud_table)
type_table = df["type"].value_counts(dropna=False).rename_axis("type").reset_index(name="count")
display(type_table)
inspection_summary = {
    "rows": int(len(df)), "columns": int(df.shape[1]),
    "duplicate_rows": duplicate_count,
    "missing_values": {c: int(v) for c, v in df.isna().sum().items()},
    "fraud_counts": {str(k): int(v) for k, v in fraud_counts.items()},
    "transaction_type_counts": {str(r["type"]): int(r["count"]) for _, r in type_table.iterrows()}
}


Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
3737323,278,CASH_IN,330218.42,C632336343,20866.00,351084.42,C834976624,452419.57,122201.15,0,0
264914,15,PAYMENT,11647.08,C1264712553,30370.00,18722.92,M215391829,0.00,0.00,0,0
85647,10,CASH_IN,152264.21,C1746846248,106589.00,258853.21,C1607284477,201303.01,49038.80,0,0
5899326,403,TRANSFER,1551760.63,C333676753,0.00,0.00,C1564353608,3198359.45,4750120.08,0,0
2544263,206,CASH_IN,78172.30,C813403091,2921331.58,2999503.88,C1091768874,415821.90,337649.60,0,0


,dtype,missing_values
step,int64,0
type,category,0
amount,float64,0
nameOrig,str,0
oldbalanceOrg,float64,0
newbalanceOrig,float64,0
nameDest,str,0
oldbalanceDest,float64,0
newbalanceDest,float64,0
isFraud,int8,0


Exact duplicate rows: 0


,label,count,percentage
0,Not fraud,6354407,99.870918
1,Fraud,8213,0.129082


,type,count
0,CASH_OUT,2237500
1,PAYMENT,2151495
2,CASH_IN,1399284
3,TRANSFER,532909
4,DEBIT,41432


## Cell 10

### Visualize transaction types and fraud labels
Both required charts use `plotly.express.pie()`. The fraud slice can be very small:
this class imbalance lets an always-non-fraud model achieve high accuracy while
catching no fraud. We therefore measure fraud precision, recall and F1 as well.
Charts describe the data; they are not used to select parameters from the test set.

In [5]:
# Cell 11
type_chart = px.pie(type_table, names="type", values="count", title="Transaction-type distribution")
type_chart.update_traces(textinfo="percent+label")
type_chart.show()
fraud_chart = px.pie(
    fraud_table, names="label", values="count", title="Fraud / non-fraud distribution",
    color="label", color_discrete_map={"Not fraud": "#3399cc", "Fraud": "#dd5544"}
)
fraud_chart.update_traces(textinfo="percent+label")
fraud_chart.show()
display(Markdown(
    f"Fraud represents **{fraud_counts.loc[1] / len(df):.4%}** of transactions. "
    f"Always predicting 'Not fraud' would be correct on **{fraud_counts.loc[0] / len(df):.4%}** "
    "of the full dataset, but its fraud recall would be **0%**. "
    "The held-out baseline below measures this on the test partition."
))


Fraud represents **0.1291%** of transactions. Always predicting 'Not fraud' would be correct on **99.8709%** of the full dataset, but its fraud recall would be **0%**. The held-out baseline below measures this on the test partition.

## Cell 12

### Define inputs, exclusions and the target
The model uses these five raw input columns:

| Feature | Meaning |
|---|---|
| `step` | Simulation time step (one hour) |
| `type` | Transaction category: CASH_IN, CASH_OUT, DEBIT, PAYMENT or TRANSFER |
| `amount` | Transaction amount |
| `oldbalanceOrg` | Origin account balance before the transaction |
| `oldbalanceDest` | Destination account balance before the transaction |

The target is `isFraud`: **1 = Fraud, 0 = Not fraud**.
Exclude `nameOrig` and `nameDest` because identifiers can encourage memorizing
customers rather than learning transferable patterns. Exclude `isFlaggedFraud`
because it is an existing rule-based flag, not an independent transaction input;
we want to evaluate our own detector without relying on that rule.

Exclude `newbalanceOrig` and `newbalanceDest`: they are recorded **after** a
transaction. They can reveal its outcome and would not be available for a
pre-transaction decision. They could be useful for a separately defined
post-transaction investigation task, but would change the deployment assumptions.
Even the old destination balance may not be accessible to a real payment service.

Remove exact full-row duplicates before splitting so identical records do not
appear in both partitions. This deterministic rule learns no data statistics.
Different transactions can share the same selected inputs; that alone is not a
reason to delete them. Free the larger original table once the inputs are copied.

In [6]:
# Cell 13
numeric_features = ["step", "amount", "oldbalanceOrg", "oldbalanceDest"]
categorical_features = ["type"]
feature_columns = numeric_features + categorical_features
excluded_features = ["nameOrig", "nameDest", "isFlaggedFraud", "newbalanceOrig", "newbalanceDest"]
if duplicate_count:
    df = df.drop_duplicates().reset_index(drop=True)
X = df.loc[:, feature_columns].copy()
# Object dtype gives the categorical imputer standard string values and np.nan.
X["type"] = X["type"].astype(object).where(df["type"].notna(), np.nan)
y = df["isFraud"].copy()
modeling_rows = len(X)
display(X.head())
print("Target counts:", y.value_counts().sort_index().to_dict())
del df


,step,amount,oldbalanceOrg,oldbalanceDest,type
0,1,9839.64,170136.0,0.0,PAYMENT
1,1,1864.28,21249.0,0.0,PAYMENT
2,1,181.00,181.0,0.0,TRANSFER
3,1,181.00,181.0,21182.0,CASH_OUT
4,1,11668.14,41554.0,0.0,PAYMENT


Target counts: {0: 6354407, 1: 8213}


## Cell 14

### Reserve an untouched 20% test set
`train_test_split` uses 80% for training and 20% for testing. `random_state=42`
makes the split reproducible; `stratify=y` approximately preserves both class
proportions. We do not select hyperparameters or thresholds using test results.
For future improvements, use cross-validation or a validation subset within the
training partition. A random split is required here; a chronological split would
better measure performance on future transactions.

In [7]:
# Cell 15
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
print("Training inputs:", X_train.shape, "Testing inputs:", X_test.shape)
split_table = pd.DataFrame({
    "train_count": y_train.value_counts().reindex([0, 1], fill_value=0),
    "test_count": y_test.value_counts().reindex([0, 1], fill_value=0)
})
split_table.index.name = "isFraud"
display(split_table)
del X, y


Training inputs: (5090096, 5) Testing inputs: (1272524, 5)


,train_count,test_count
isFraud,,
0,5083526,1270881
1,6570,1643


## Cell 16

### Preprocess safely and choose a decision tree
Numeric missing values use the **training median**. Missing transaction types
use the **training most frequent type**. One-hot encoding creates a separate
binary column for each type without inventing an order; `handle_unknown='ignore'`
allows a new type at prediction time. A new type still deserves monitoring.

A `ColumnTransformer` applies each transformation to the right columns. A
`Pipeline` fits preprocessing only during `model.fit(X_train, y_train)` and reuses
it for testing and examples, preventing leakage from fitted preprocessing.
See [scikit-learn's leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html).

A decision tree learns a sequence of if/else splits from labeled training data,
greedily reducing weighted Gini impurity. It suits mixed numeric/categorical
tabular features after encoding, captures nonlinear rules, and needs no scaling.
These parameters are fixed **before** evaluating the test set:

- `criterion='gini'`: choose splits that make child nodes purer.
- `max_depth=8`: cap complexity and help limit overfitting.
- `min_samples_leaf=20`: avoid leaves supported by only a few rows.
- `class_weight='balanced'`: give the rare fraud class more weight, using training
  frequencies; this can improve recall while increasing false alarms.
- `random_state=42`: reproducible tree randomness.

These are reasonable educational choices, not claimed optimal settings.
[DecisionTreeClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html).

In [8]:
# Cell 17
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
tree_parameters = {
    "criterion": "gini", "max_depth": 8, "min_samples_leaf": 20,
    "class_weight": "balanced", "random_state": RANDOM_SEED
}
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(**tree_parameters))
])
model.fit(X_train, y_train)
print("Training complete.")
print("Tree depth:", model.named_steps["classifier"].get_depth())
print("Tree leaves:", model.named_steps["classifier"].get_n_leaves())
print("Encoded features:", model.named_steps["preprocessor"].get_feature_names_out().tolist())


Training complete.
Tree depth: 8
Tree leaves: 67
Encoded features: ['numeric__step', 'numeric__amount', 'numeric__oldbalanceOrg', 'numeric__oldbalanceDest', 'categorical__type_CASH_IN', 'categorical__type_CASH_OUT', 'categorical__type_DEBIT', 'categorical__type_PAYMENT', 'categorical__type_TRANSFER']


## Cell 18

### Evaluate once and compare to an always-non-fraud baseline
`model.score(X_test, y_test)` explicitly returns test accuracy. `model.predict()`
provides predicted labels. Fraud (`1`) is the positive class:

- **Precision:** of flagged transactions, how many are truly fraud?
- **Recall:** of actual frauds, how many did we catch?
- **F1:** harmonic mean of precision and recall.
- **Confusion matrix:** rows = true labels, columns = predicted labels, ordered
  Not fraud then Fraud; cells are TN, FP / FN, TP.

False positives interrupt legitimate payments; false negatives miss fraud.
`zero_division=0` reports precision/F1 as zero when a model predicts no fraud.
The baseline always predicts `0`, regardless of the inputs. Neither this
comparison nor the following outputs is used to retrain or tune the tree.

In [9]:
# Cell 19
test_accuracy = float(model.score(X_test, y_test))
y_pred = model.predict(X_test)
baseline = DummyClassifier(strategy="constant", constant=0)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

def evaluate_predictions(truth, predictions, accuracy):
    return {
        "accuracy": float(accuracy),
        "precision": float(precision_score(truth, predictions, pos_label=1, zero_division=0)),
        "recall": float(recall_score(truth, predictions, pos_label=1, zero_division=0)),
        "f1": float(f1_score(truth, predictions, pos_label=1, zero_division=0)),
        "confusion_matrix": confusion_matrix(truth, predictions, labels=[0, 1]).tolist()
    }

tree_metrics = evaluate_predictions(y_test, y_pred, test_accuracy)
baseline_metrics = evaluate_predictions(y_test, baseline_pred, baseline.score(X_test, y_test))
metrics_table = pd.DataFrame({
    "Decision tree": {k: v for k, v in tree_metrics.items() if k != "confusion_matrix"},
    "Always Not fraud": {k: v for k, v in baseline_metrics.items() if k != "confusion_matrix"}
}).T
display(metrics_table.style.format("{:.6f}"))
for name, result in [("Decision tree", tree_metrics), ("Always Not fraud", baseline_metrics)]:
    print(name, "confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(result["confusion_matrix"],
        index=["True Not fraud", "True Fraud"],
        columns=["Predicted Not fraud", "Predicted Fraud"]))

cm = np.asarray(tree_metrics["confusion_matrix"])
cm_chart = px.imshow(
    cm, x=["Not fraud", "Fraud"], y=["Not fraud", "Fraud"],
    text_auto=True, color_continuous_scale="Blues",
    labels={"x": "Predicted label", "y": "True label", "color": "Count"},
    title="Decision tree: test confusion matrix"
)
cm_chart.show()
tn, fp, fn, tp = cm.ravel()
display(Markdown(
    f"The tree catches **{tp:,}** frauds and misses **{fn:,}**, with **{fp:,}** "
    f"false alarms. Its fraud precision is **{tree_metrics['precision']:.2%}**, "
    f"recall **{tree_metrics['recall']:.2%}**, and F1 **{tree_metrics['f1']:.4f}**. "
    f"The baseline has **{baseline_metrics['accuracy']:.2%}** accuracy but "
    "**0% fraud recall**. Accuracy alone therefore does not describe fraud detection quality."
))


,accuracy,precision,recall,f1
Decision tree,0.952175,0.026166,0.995131,0.050991
Always Not fraud,0.998709,0.000000,0.000000,0.000000


Decision tree confusion matrix (rows=true, columns=predicted):


,Predicted Not fraud,Predicted Fraud
True Not fraud,1210030,60851
True Fraud,8,1635


Always Not fraud confusion matrix (rows=true, columns=predicted):


,Predicted Not fraud,Predicted Fraud
True Not fraud,1270881,0
True Fraud,1643,0


The tree catches **1,635** frauds and misses **8**, with **60,851** false alarms. Its fraud precision is **2.62%**, recall **99.51%**, and F1 **0.0510**. The baseline has **99.87%** accuracy but **0% fraud recall**. Accuracy alone therefore does not describe fraud detection quality.

## Cell 20

### Make an illustrative prediction
No user transaction was supplied. The following **invented illustrative input**
has no known true label; its output is a model prediction, not proof of fraud.
Build a one-row DataFrame with exactly the same raw columns, in the same order
as training. Calling `model.predict()` applies the already-fitted preprocessing
and tree together. Do not fit an encoder or imputer again on the example.

In [10]:
# Cell 21
example_values = {
    "step": 100, "amount": 200000.0,
    "oldbalanceOrg": 200000.0, "oldbalanceDest": 0.0,
    "type": "TRANSFER"
}
example = pd.DataFrame([example_values], columns=feature_columns)
assert list(example.columns) == list(X_train.columns)
example_prediction = int(model.predict(example)[0])
example_label = "Fraud" if example_prediction == 1 else "Not fraud"
display(example)
print("Illustrative example prediction:", example_label)
example_result = {
    "illustrative": True, "known_true_label": None,
    "features": example_values, "prediction": example_prediction,
    "prediction_label": example_label
}


,step,amount,oldbalanceOrg,oldbalanceDest,type
0,100,200000.0,200000.0,0.0,TRANSFER


Illustrative example prediction: Fraud


## Cell 22

### Save results locally and verify them
Save metrics, split settings, dataset information, package versions and the
illustrative prediction as JSON; also save a compact comparison CSV. Files are
written relative to the notebook's working directory, and their contents remain
visible below. Rerunning replaces these result files. The small checks verify
split sizes, valid predictions and saved-file consistency without tuning a model.

In [11]:
# Cell 23
evaluation_results = {
    "dataset_handle": DATASET_HANDLE, "csv_filename": csv_path.name,
    "inspection": inspection_summary, "modeling_rows": modeling_rows,
    "feature_columns": feature_columns, "excluded_features": excluded_features,
    "target": "isFraud", "positive_label": 1,
    "split": {"train_rows": len(X_train), "test_rows": len(X_test),
              "test_size": 0.20, "stratified": True, "random_state": RANDOM_SEED},
    "tree_parameters": tree_parameters, "package_versions": package_versions,
    "decision_tree": tree_metrics, "always_non_fraud_baseline": baseline_metrics,
    "confusion_matrix_order": [0, 1],
    "test_set_used_for_tuning": False
}
metrics_path = OUTPUT_DIR / "evaluation_metrics.json"
example_path = OUTPUT_DIR / "example_prediction.json"
comparison_path = OUTPUT_DIR / "metrics_comparison.csv"
metrics_path.write_text(json.dumps(evaluation_results, indent=2), encoding="utf-8")
example_path.write_text(json.dumps(example_result, indent=2), encoding="utf-8")
metrics_table.to_csv(comparison_path, index_label="model")

assert len(X_train) + len(X_test) == modeling_rows
assert X_train.index.intersection(X_test.index).empty
assert set(np.unique(y_pred)).issubset({0, 1})
assert int(cm.sum()) == len(y_test)
assert tree_metrics["accuracy"] == float(np.mean(y_pred == y_test.to_numpy()))
assert baseline_metrics["recall"] == 0.0
assert json.loads(metrics_path.read_text(encoding="utf-8")) == evaluation_results
assert json.loads(example_path.read_text(encoding="utf-8")) == example_result
print("Run checks passed.")
for path in [metrics_path, example_path, comparison_path]:
    print("Saved:", path)
print(metrics_path.read_text(encoding="utf-8"))
print(example_path.read_text(encoding="utf-8"))
display(pd.read_csv(comparison_path))


Run checks passed.
Saved: C:\Users\jonat\OneDrive\Desktop\St1 ML\outputs\evaluation_metrics.json
Saved: C:\Users\jonat\OneDrive\Desktop\St1 ML\outputs\example_prediction.json
Saved: C:\Users\jonat\OneDrive\Desktop\St1 ML\outputs\metrics_comparison.csv
{
  "dataset_handle": "ealaxi/paysim1",
  "csv_filename": "PS_20174392719_1491204439457_log.csv",
  "inspection": {
    "rows": 6362620,
    "columns": 11,
    "duplicate_rows": 0,
    "missing_values": {
      "step": 0,
      "type": 0,
      "amount": 0,
      "nameOrig": 0,
      "oldbalanceOrg": 0,
      "newbalanceOrig": 0,
      "nameDest": 0,
      "oldbalanceDest": 0,
      "newbalanceDest": 0,
      "isFraud": 0,
      "isFlaggedFraud": 0
    },
    "fraud_counts": {
      "0": 6354407,
      "1": 8213
    },
    "transaction_type_counts": {
      "CASH_OUT": 2237500,
      "PAYMENT": 2151495,
      "CASH_IN": 1399284,
      "TRANSFER": 532909,
      "DEBIT": 41432
    }
  },
  "modeling_rows": 6362620,
  "feature_columns": [
  

,model,accuracy,precision,recall,f1
0,Decision tree,0.952175,0.026166,0.995131,0.050991
1,Always Not fraud,0.998709,0.000000,0.000000,0.000000


## Cell 24

### Limitations and next steps
This is an educational model on synthetic data, not a production payment filter.
A random split can share customers and time patterns across partitions even after
identifiers are excluded; it may overestimate future performance. Simulation
artifacts, incomplete destination balances, changes in fraud behavior and unseen
transaction types also limit generalization. Class weighting can cause many false
alarms, and a bounded tree can miss complex patterns. Predictions are not calibrated
risk estimates. The illustrative example has no ground truth.

For a later project, validate chronologically and by customer, assess business
costs, and choose settings only inside the training data before evaluating a
fresh holdout. Those experiments are outside this assignment's implementation.

### Submission and requirements checklist
- [x] Install required packages and load KaggleHub's downloaded CSV portably.
- [x] Inspect samples, shape, dtypes, missing values, duplicates and label counts.
- [x] Create both Plotly pie charts and explain imbalance.
- [x] Explain features/target/exclusions, missing values, encoding and leakage prevention.
- [x] Implement reproducible stratified 80/20 split and decision tree training.
- [x] Use `model.score()` and fraud precision/recall/F1/confusion matrices plus a baseline.
- [x] Use `model.predict()` on a clearly illustrative, matching input.
- [x] Save local JSON/CSV results and show outputs; include execution checks.
- [x] Provide README with setup, explanations, results and submission instructions.
- [x] Execution: all 11 code cells completed in a fresh kernel on 2026-11-02;
      the final code cell printed “Run checks passed”. Results are saved in the notebook and README.
- [ ] Compare with the original assignment attachment, which was unavailable here.
- [ ] Manually commit on a feature branch, push and open a GitHub pull request.
- [ ] Manually submit the pull request URL to TUMO Labs.
- [ ] Manual peer review 1: review one other participant's submission.
- [ ] Manual peer review 2: review a second participant's submission.